## Model Selection

## Change this we are doing SHAP first
Before we move on we would like to choose the best possible model for each of the two cases: daily forecast and hourly forecast. It seems that for the daily forecast in the long term you choose linear_order2 in the short to medium term potentially hybrid_order2. For the hourly forecast it seems that just using XGBoost is the best possible model.

The main question is that in the daily forecast if we were to optimise the hyperparameters of the XGBoost model within the hybrid model would it make hybrid_order2 better than linear_order2 in both the short and long term. It would also be nice to do hyperparamter optimisation for the hourly forecast as well just to see whether we can improve the predictions or not. 

Finally it would be good to look at SHAP values to see if we can drop any of the features, particualarly some of the lags as they can make the models computationally expensive.

For our own use (maybe delete later): http://kaggle.com/code/prashant111/a-guide-on-xgboost-hyperparameters-tuning

https://hyperopt.github.io/hyperopt/?source=post_page

https://github.com/hyperopt/hyperopt/wiki/FMin

In [1]:
from hyperopt import hp, fmin, tpe, hp, Trials
from hyperopt.pyll import scope
from jfk_taxis import load_design, load_config, load_models, load_lags, create_val_data, wrapped_objective, save_hyperparams, load_hyperparams, save_obj, load_obj, split_params, test_hyperparams, load_ts_data, split_test_train_sets, load_process_lags, compute_shap_values, shap_plots, extract_top_x_features_dict 
from sklearn.linear_model import LinearRegression
import numpy as np
import shap

In [2]:
# Load config and project root
config, PROJECT_ROOT = load_config()


In [3]:
# First load the significant lags
daily_lags, used_hourly_lags = load_process_lags()

In [4]:
# Load ts data
ts_daily, ts_hourly = load_ts_data()

In [5]:
# Create test train sets as defined in config.yml
ts_daily_train, ts_daily_test, ts_hourly_train, ts_hourly_test = split_test_train_sets(ts_daily, ts_hourly)

## SHAP values

We now seek to explore the shap values of the models. We will first look at two example models one daily one hourly to get a feel for the SHAP values before we compute them all. 

In [6]:
# First we assemble the design and model signatures from config.yml
DAILY_LINEAR_SIG = config["model_sigs"]["daily_linear"]
DAILY_HYBRID_SIG = config["model_sigs"]["daily_hybrid"]
HOURLY_LINEAR_SIG = config["model_sigs"]["hourly_linear"]
HOURLY_HYBRID_SIG = config["model_sigs"]["hourly_hybrid"]

# Now we get the model prefixes from config.yml
DAILY_LINEAR_PREFIX = config["model_naming"]["linear_model_prefix"] + "2"
DAILY_HYBRID_PREFIX = config["model_naming"]["hybrid_model_prefix"] + "2"
HOURLY_LINEAR_PREFIX = config["model_naming"]["linear_model_prefix"] + "2"
HOURLY_HYBRID_PREFIX = config["model_naming"]["hybrid_model_prefix"] + "2"
DEFAULT_NON_LINEAR_PREFIX = config["model_naming"]["default_non_linear"]

# Daily prefix
DAILY_PREFIX = config["shap"]["daily_prefix"]

# Hourly prefix 
HOURLY_PREFIX = config["shap"]["hourly_prefix"]

# Titles for plots and to key values for dicts
DAILY_LINEAR_NAME = f"{DAILY_PREFIX}_{DAILY_LINEAR_PREFIX}"
DAILY_HYBRID_NAME = f"{DAILY_PREFIX}_{DAILY_HYBRID_PREFIX}"
DAILY_NON_LINEAR_NAME = f"{DAILY_PREFIX}_{DEFAULT_NON_LINEAR_PREFIX}"
HOURLY_LINEAR_NAME = f"{HOURLY_PREFIX}_{HOURLY_LINEAR_PREFIX}"
HOURLY_HYBRID_NAME = f"{HOURLY_PREFIX}_{HOURLY_HYBRID_PREFIX}"
HOURLY_NON_LINEAR_NAME = f"{HOURLY_PREFIX}_{DEFAULT_NON_LINEAR_PREFIX}"

# Dictionary to store the shap values (it will contain tuples of shap values and design matricies)
shap_values_dict = {}


In [ ]:
# Compute the SHAP values for daily non linear
shap_values, X = compute_shap_values(DAILY_LINEAR_SIG, DAILY_LINEAR_SIG, DEFAULT_NON_LINEAR_PREFIX, linear= False, hybrid= False)

In [ ]:
# Create summary plot and bar plot by mean absolute value
shap_plots(shap_values, X, DAILY_NON_LINEAR_NAME)

In [ ]:
# Store shap values and design matrix in dict
shap_values_dict[DAILY_NON_LINEAR_NAME] = (shap_values, X)

As you can see in the above SHAP summary plot, lag_1 has the widest spread with low feature values causing low impact on model output and vice versa. It is interesting that lag_1 is the most significant feature by SHAP value as during testing I tried several different naive baselines, like lag of 1, lag of 7 etc etc and lag of 1 was consitently the best which is supported here by our SHAP values. 

Interestingly you also see that lag_7 and lag_365 have reasonably wide spreads again lining up with our EDA that there was both weekly and yearly seasonality in the taxi data. Now this is an XGBoost model so it is interesting that it has picked up on the lags more rather than the fourier features when looking at seasonality. Although the model does still make use of the fourier features. 

It's worth looking at a daily linear model to see if it favours the fourier features over the lags.

In [ ]:
# Compute SHAP values for daily linear_order2
shap_values, X = compute_shap_values(DAILY_LINEAR_SIG, DAILY_LINEAR_SIG, DAILY_LINEAR_PREFIX, linear= True, hybrid= False)

In [ ]:
# Create summary plot and bar plot by mean absolute value
shap_plots(shap_values, X, DAILY_LINEAR_NAME)

In [ ]:
# Store shap values and design matrix in dict
shap_values_dict[DAILY_LINEAR_NAME] = (shap_values, X)

The above is super interesting because it shows that our fourier features aren't at the top at all here. This goes against what I expected as I orignally thought that the linear regression would like the smooth fourier features that could allow it to capture things like annual seasonality. But it seems it has used the lags like lag_364 and lag_7 to capture the seasonality instead. 

This seems to be more of a memory property of the data, as in the value last year affects the value this year more than the yearly up and down of say slightly more taxis around the holidays (due to more flights). So lag features like lag 364 can better capture both the seasanolity of 1 year but also the year on year effects, like high values last year means high values this year is likely. 

There are a couple interesting things to note in terms of the shape on the summary plots. For example for lag 2 it seems high values 2 days ago cause a decrease in the model prediction and vice versa. But high values one day ago strongly suggest high values the next day. Lag 7 and lag 14 have high mean absolute shap values which is again are weekly seasonality. 

The other thing that is super intersting is the trend doesn't seem to appear in the top 30. This suggests it's not significant for the model. However this is slightly suspicious as from our inital modelling it seems that linear_order2 outperformed the other linear_order models, which would suggest that the trend had some effect on the outcome particualarly long term. So it may be worth adding the trend back in particularly for long term forecasts.

Finally it would be interesting to have an inital look at the SHAP values for the XGBoost component of the hybrid models, just to see what they are picking up. We will do this for hybrid_order2.

In [ ]:
# Compute SHAP values for daily hybrid_order2
shap_values, X = compute_shap_values(DAILY_HYBRID_SIG, DAILY_HYBRID_SIG, DAILY_HYBRID_PREFIX, linear= True, hybrid= True)

In [ ]:
# Create summary plot and bar plot by mean absolute value
shap_plots(shap_values, X, DAILY_HYBRID_NAME)

In [ ]:
# Store shap values and design matrix in dict
shap_values_dict[DAILY_HYBRID_NAME] = (shap_values, X)

The above is intersting. It would seem trend is the most important feature according to mean absolute SHAP value. There is a good actually a good plausible explanation for this. Notice that high values of trend cause a decrease in model predictions. Well if you look at the actual daily taxi data plotted you notice that since COVID (so high values for trend) the daily taxi count has decreased. So this is potentially what this XGBoost model is capturing that difference between pre and post COVID. You can see this again with the fact that it has marked the lag 364 as a negative predictor, so high values last year imply low values this year. This again is likely from that huge drop in taxi count from COVID as well as the general overall decline in yellow taxi usage that we see leading up to COVID due to the introduction of things like Green taxis + Uber from our EDA (although these are just intersting guesses).

It's also very clear that these effects are much smaller than that of the underlying model (hybrid_order2s linear component is the same as for linear_order2) with trend only having mean absoulute SHAP value of 59.61 compared to the linear components 1112.07 for y_lag_1.

We now do the same for the hourly models.



In [ ]:
# Compute the SHAP values for hourly non linear
shap_values, X = compute_shap_values(HOURLY_LINEAR_SIG, HOURLY_LINEAR_SIG, DEFAULT_NON_LINEAR_PREFIX, linear= False, hybrid= False)

In [ ]:
# Create summary plot and bar plot by mean absolute value
shap_plots(shap_values, X, HOURLY_NON_LINEAR_NAME)

In [ ]:
# Store shap values and design matrix in dict
shap_values_dict[HOURLY_NON_LINEAR_NAME] = (shap_values, X)

In [ ]:
# Compute the SHAP values for hourly linear_order2
shap_values, X = compute_shap_values(HOURLY_LINEAR_SIG, HOURLY_LINEAR_SIG, HOURLY_LINEAR_PREFIX, linear= True, hybrid= False)

In [ ]:
# Create summary plot and bar plot by mean absolute value
shap_plots(shap_values, X, HOURLY_LINEAR_NAME)

In [ ]:
# Store shap values and design matrix in dict
shap_values_dict[HOURLY_LINEAR_NAME] = (shap_values, X)

In [ ]:
# Compute the SHAP values for hourly hybrid_order2
shap_values, X = compute_shap_values(HOURLY_HYBRID_SIG, HOURLY_HYBRID_SIG, HOURLY_HYBRID_PREFIX, linear= True, hybrid= True)

In [ ]:
# Create summary plot and bar plot by mean absolute value
shap_plots(shap_values, X, HOURLY_HYBRID_NAME)

In [ ]:
# Store shap values and design matrix in dict
shap_values_dict[HOURLY_HYBRID_NAME] = (shap_values, X)

In [ ]:
# Save shap values dict
save_obj(shap_values_dict, config["saving"]["shap_values_file"])

What we immediately observe is we have the same pattern from the daily models. The daily and weekly seasonality (24 and 168 hours) both appear as lags rather than as their respective fourier features. We also see that once again lag 1 is the most important feature (by mean abs SHAP value) to both the linear and non linear models. 

We also see that the lag of 336 (168 * 2) also has high importance which again supports the idea that the lags are picking up the seasonality perhaps better than the fourier features. 

We further see that lag 8736 is significant in all three models, this is our yearly seasonality in the data. Although not flagged in the EDA (potentially because we tried 8760 for exactly one year, although this does show up in the hybrid). If you read config.yml you can see we included some additional significant lags around 8760 and 4380 to try and capture any potentially half or annual seasonlaity with the lags. We observe at least in the inital top 20 we can't see any half annual seasonality. But can see the annual lags.

What we would now like to do is select the top 30 features by mean absoulute SHAP value and retrain the models, to see if they caputre most of the signal. 

For the fourier features if we have at least one component (say just cos(2,freq=W-SUN)) then we will pass the entire weekly fourier feature, this is because there are likely interactions especially with say the sin(2,freq=W-SUN) feature and so removing them could mean this feature doesn't capture the same amount of signal.



In [7]:
# Load shap values dict
shap_values_dict = load_obj(config["saving"]["shap_values_file"])

In [8]:
# Constant for number of top features to extract
TOP_X = config["shap"]["num_features_to_extract"]

In [9]:
# We extract the top X features for each model and in a new dict with the same keys as shap_values_dict but the values are now (list of lags, list of fourier features and list of trends)
top_x_features_dict = extract_top_x_features_dict(shap_values_dict, x = TOP_X)

ValueError: too many values to unpack (expected 2)

In [10]:
linear_design_loaded, non_linear_design_loaded = load_design(config["model_sigs"]["daily_linear"])


In [11]:
linear_design_loaded["linear_order2"][0]

,const,trend,trend_squared,"sin(1,freq=YE-DEC)","cos(1,freq=YE-DEC)","sin(2,freq=YE-DEC)","cos(2,freq=YE-DEC)","sin(3,freq=YE-DEC)","cos(3,freq=YE-DEC)","sin(4,freq=YE-DEC)",...,y_lag_329,y_lag_330,y_lag_333,y_lag_335,y_lag_336,y_lag_343,y_lag_350,y_lag_357,y_lag_364,y_lag_371
pickup_date,,,,,,,,,,,,,,,,,,,,,
2012-01-07,1.0,372.0,138384.0,0.102821,0.994700,0.204552,0.978856,0.304115,0.952635,0.400454,...,5106.0,5898.0,5032.0,5551.0,4026.0,4774.0,4880.0,4970.0,5459.0,2539.0
2012-01-08,1.0,373.0,139129.0,0.119881,0.992788,0.238033,0.971257,0.352752,0.935717,0.462383,...,6332.0,5106.0,4716.0,6199.0,5551.0,5934.0,6208.0,5471.0,7030.0,3438.0
2012-01-09,1.0,374.0,139876.0,0.136906,0.990584,0.271234,0.962513,0.400454,0.916317,0.522133,...,6448.0,6332.0,5537.0,5032.0,6199.0,6589.0,6303.0,7486.0,7404.0,8304.0
2012-01-10,1.0,375.0,140625.0,0.153891,0.988088,0.304115,0.952635,0.447094,0.894487,0.579421,...,5545.0,6448.0,5898.0,4716.0,5032.0,2966.0,5350.0,6704.0,5276.0,7369.0
2012-01-11,1.0,376.0,141376.0,0.170830,0.985301,0.336637,0.941634,0.492548,0.870285,0.633978,...,5331.0,5545.0,5106.0,5537.0,4716.0,4032.0,4131.0,5498.0,3222.0,6155.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-27,1.0,4744.0,22505536.0,-0.085965,0.996298,-0.171293,0.985220,-0.255353,0.966848,-0.337523,...,4038.0,4094.0,3897.0,4572.0,3917.0,4553.0,4629.0,6006.0,6283.0,4852.0
2023-12-28,1.0,4745.0,22515025.0,-0.068802,0.997630,-0.137279,0.990532,-0.205104,0.978740,-0.271958,...,3972.0,4038.0,5057.0,4306.0,4572.0,4366.0,5210.0,5617.0,6597.0,5132.0
2023-12-29,1.0,4746.0,22524516.0,-0.051620,0.998667,-0.103102,0.994671,-0.154309,0.988023,-0.205104,...,4318.0,3972.0,4850.0,3897.0,4306.0,4761.0,5780.0,4877.0,6218.0,5077.0


In [ ]:
# Define search space
space = {
    # We rely on early stopping when fitting so this isn't an optimised value
    # number of trees
    "n_estimators": 500,

    # Learning rate
    # step size shrinkage, smaller = slower but more precise learning
    "learning_rate": 0.05,

    # Depth/complexity
    # Max depth of tree, larger more complex trees but can cause overfitting
    "max_depth": scope.int(hp.quniform("max_depth", 3, 6, 1)), # scope.int ensures we take ints only
    # minimum "weight" needed in child node. Higher values more conservative, fewer splits helps prevent overfitting
    "min_child_weight": hp.loguniform("min_child_weight", -2.3, 2.3), # approx [0.1, 10]

    # Randomisation/feature subsampling
    # fraction of rows used per tree, lower adds randomness reduces overfitting
    "subsample": hp.uniform("subsample", 0.6, 1.0),
    # fraction of features used per tree
    "colsample_bytree": hp.uniform("colsample_bytree", 0.6, 1.0),

    # Regularisation
    # L2 penalty, good range is [0.1, 10] we use loguniform because this means that every order of magnitude has equal probability, the def of log uniform in hyperopt is that it returns a value exp(U(low,high)) where U is uniform dist.  
    "reg_lambda": hp.loguniform("reg_lambda", np.log(1e-2), np.log(100)), # [0.01, 100]
    # L1 penalty
    "reg_alpha": hp.loguniform("reg_alpha", np.log(1e-3), np.log(10)), # [0.001, 10]

    # Split pnealty (gamma) 
    # minimum loss reduction required to split a node, higher values = more conservative
    "gamma": hp.loguniform("gamma", -7.0, 2.3), # approx [0.0009, 10]

    "random_state": 37,
    #"early_stopping_rounds": 100,
    "eval_metric": "mae", 
    "tree_method": "hist",
    "device": "cuda" # Use GPU if available
}
    
    

There are a few interesting things we would like to vary when doing our Bayesian hyperparamter optimisation. The first is that we want to both include and exclude COVID from the data we use to tune hyperparamters on.

The reason for this is because during COVID the usual seasonality breaks down dramatically as we get unusal travel patterns. So it may be worth tuning a model on data without COVID as then it is being asseseed more on its ability to pick up the more "normal" patterns within the data. You then have a question of do you use pre or post COVID data, there is more pre COVID data but it will obviously be less relevant for forecasting in the present. Alternatively it may be actually be worth including the COVID data as then the model is tuned to be more robust to "unusal" regimines within the data.

So as it's not clear which of the three will be optimal we will just run all three and then compare the models that hyperopt finds.

For the daily series we wil use a 30 day forecast in our objection function to validate with. For the hourly series we will do a weekly forecast (168 hours).

In [ ]:
# Dictionary of parameters for Baysian optimisation
bayes_dict = {}

In [ ]:
# Daily non_linear pre, incl and post COVID

# Set the parameters for creat_val_data and the objective function
n_splits = config["hyperparameter_tuning"]["n_splits"]
test_size = config["hyperparameter_tuning"]["daily_test_size"]
lags = daily_lags
constant = False
order = 0
fourier_features = config["modelling"]["daily_fourier_features"]
time_step = "D"
hybrid = None
steps = config["hyperparameter_tuning"]["daily_test_size"]


# Looping through this dict avoids having to have three separate code cells, the keys are the sigs, values are the tsk
tmp_dict = {
    "daily_non_linear_pre_COVID": ts_daily_train[:"2020-01-01"],
    "daily_non_linear_incl_COVID": ts_daily_train,
    #"daily_non_linear_post_COVID": ts_daily_train["2022-01-01":] TODO decide if we want to do this, there are problems with the folds being too small
}

# Add to Bayes dict
for key, value in tmp_dict.items():
    bayes_dict[key] = {
        "n_splits" : n_splits,
        "test_size" : test_size,
        "lags" : lags,
        "constant" : constant,
        "order" : order,
        "fourier_features" : fourier_features,
        "time_step" : time_step,
        "ts" : value,
        "hybrid" : hybrid,
        "steps" : steps
    }

In [ ]:
# Daily non_linear hybrid pre, incl and post COVID

# Set the parameters for creat_val_data and the objective function
n_splits = config["hyperparameter_tuning"]["n_splits"]
test_size = config["hyperparameter_tuning"]["daily_test_size"]
lags = daily_lags
constant = True
order = 2 # we use 2nd order as this performed the best in the purely linear case, the hybrid model is just a boosted version of the purely linear case so we expect order 2 to perform the best
fourier_features = config["modelling"]["daily_fourier_features"]
time_step = "D"
hybrid = LinearRegression(fit_intercept= False)
steps = config["hyperparameter_tuning"]["daily_test_size"]


# Looping through this dict avoids having to have three separate code cells, the keys are the sigs, values are the tsk
tmp_dict = {
    "daily_hybrid_non_linear_pre_COVID": ts_daily_train[:"2020-01-01"],
    "daily_hybrid_non_linear_incl_COVID": ts_daily_train,
    #"daily_hybrid_non_linear_post_COVID": ts_daily_train["2022-01-01":]
}

# Add to Bayes dict
for key, value in tmp_dict.items():
    bayes_dict[key] = {
        "n_splits" : n_splits,
        "test_size" : test_size,
        "lags" : lags,
        "constant" : constant,
        "order" : order,
        "fourier_features" : fourier_features,
        "time_step" : time_step,
        "ts" : value,
        "hybrid" : hybrid,
        "steps" : steps
    }

In [ ]:
# Hourly non_linear pre, incl and post COVID

# Set the parameters for creat_val_data and the objective function
n_splits = config["hyperparameter_tuning"]["n_splits"]
test_size = config["hyperparameter_tuning"]["hourly_test_size"]
lags = used_hourly_lags
constant = False
order = 0
fourier_features = config["modelling"]["hourly_fourier_features"]
time_step = "h"
hybrid = None
steps = config["hyperparameter_tuning"]["hourly_test_size"]


# Looping through this dict avoids having to have three separate code cells, the keys are the sigs, values are the tsk
tmp_dict = {
    "hourly_non_linear_pre_COVID": ts_hourly_train[:"2020-01-01"],
    "hourly_non_linear_incl_COVID": ts_hourly_train,
    #"hourly_non_linear_post_COVID": ts_hourly_train["2022-01-01":]
}

# Add to Bayes dict
for key, value in tmp_dict.items():
    bayes_dict[key] = {
        "n_splits" : n_splits,
        "test_size" : test_size,
        "lags" : lags,
        "constant" : constant,
        "order" : order,
        "fourier_features" : fourier_features,
        "time_step" : time_step,
        "ts" : value,
        "hybrid" : hybrid,
        "steps" : steps
    }


In [ ]:
# Hourly non_linear hybrid pre, incl and post COVID

# Set the parameters for creat_val_data and the objective function
n_splits = config["hyperparameter_tuning"]["n_splits"]
test_size = config["hyperparameter_tuning"]["hourly_test_size"]
lags = used_hourly_lags
constant = True
order = 2 # we use 2nd order as this performed the best in the purely linear case, the hybrid model is just a boosted version of the purely linear case so we expect order 2 to perform the best
fourier_features = config["modelling"]["hourly_fourier_features"]
time_step = "h"
hybrid = LinearRegression(fit_intercept= False)
steps = config["hyperparameter_tuning"]["hourly_test_size"]


# Looping through this dict avoids having to have three separate code cells, the keys are the sigs, values are the tsk
tmp_dict = {
    "hourly_hybrid_non_linear_pre_COVID": ts_hourly_train[:"2020-01-01"],
    "hourly_hybrid_non_linear_incl_COVID": ts_hourly_train,
    #"hourly_hybrid_non_linear_post_COVID": ts_hourly_train["2022-01-01":]
}

# Add to Bayes dict
for key, value in tmp_dict.items():
    bayes_dict[key] = {
        "n_splits" : n_splits,
        "test_size" : test_size,
        "lags" : lags,
        "constant" : constant,
        "order" : order,
        "fourier_features" : fourier_features,
        "time_step" : time_step,
        "ts" : value,
        "hybrid" : hybrid,
        "steps" : steps
    }


In [ ]:
# Loop through the Bayes dict, create the folds and run fmin for 100 evals to optimise hyperparamters, save the hyperparameters and sigs to a .pkl object

# list to store the keys
sig_list = []

for key, value in bayes_dict.items():
    print(f"Running hyperparameter optimisation for {key}")
    
    # Create the folds
    fold_dict = create_val_data(value["n_splits"], value["test_size"], value["lags"], value["constant"], value["order"], value["fourier_features"], value["time_step"], value["ts"])

    # Set attributes of wrapped_objective
    wrapped_objective.fold_dict = fold_dict
    wrapped_objective.lags = value["lags"]
    wrapped_objective.steps = value["steps"]
    wrapped_objective.hybrid = value["hybrid"]

    # Optimisation algorithm
    trials = Trials()

    best_hyperparams = fmin(fn = wrapped_objective,
                            space = space,
                            algo = tpe.suggest,
                            max_evals = config["hyperparameter_tuning"]["max_evals"],
                            trials = trials)

    # Log hyperparams 
    print("The best hyperparamters are: ", "\n")
    print(best_hyperparams)

    # Before we save the hyperparams we need to change the type of max_alpha to int 
    # as well as add some of the parameters that aren't in best_hyperparmas, n_estimators, learnining_rate, random_state, eval_metric, tree_method and device
    # At the same time we may as well convert the remaining hyperparams to floats rather than np.float64 
    for key2, value2 in best_hyperparams.items():
        if key2 != "max_depth":
            best_hyperparams[key2] = float(value2)
        else:
            best_hyperparams[key2] = int(value2)

    best_hyperparams["n_estimators"] = space["n_estimators"]
    best_hyperparams["learning_rate"] = space["learning_rate"]
    best_hyperparams["random_state"] = space["random_state"]
    best_hyperparams["eval_metric"] = space["eval_metric"]
    best_hyperparams["tree_method"] = space["tree_method"]
    best_hyperparams["device"] = space["device"]

    # Save hyperparams
    save_hyperparams(best_hyperparams, key)

    # Append the sig to the list
    sig_list.append(key)

# Save signatures
save_obj(sig_list, "hyperparam_sigs")

In [ ]:
# Load signatures
hyper_sig = load_obj("hyperparam_sigs")

# Dict of hyperparams
hyper_dict = {}

# Load the hyperparams for each model
for sig in hyper_sig:
    hyper_params = load_hyperparams(sig)
    hyper_dict[sig] = hyper_params


In [ ]:
# The first interesting thing to compare would be how do the models tuned on pre, incl and post COVID compare to each other as well as to the previous best model from the modelling notebook
# to make this easier we will split up into four dicts, daily non-linear, daily hybrid, hourly non-linear, hourly hybrid

daily_non_linear_dict = {}
daily_hybrid_dict = {}
hourly_non_linear_dict = {}
hourly_hybrid_dict = {}

# Split into the four dicts
daily_non_linear_dict, daily_hybrid_dict, hourly_non_linear_dict, hourly_hybrid_dict = split_params(hyper_dict)


# Create dict of these dicts to pass to test_hyperparams
dict_full = {
    config["hyperparameter_tuning"]["daily_linear_key"]: daily_non_linear_dict, 
    config["hyperparameter_tuning"]["daily_hybrid_key"]: daily_hybrid_dict, 
    config["hyperparameter_tuning"]["hourly_linear_key"]: hourly_non_linear_dict, 
    config["hyperparameter_tuning"]["hourly_hybrid_key"]: hourly_hybrid_dict
    }

In [ ]:
# We are now going to forecast all of the models inside these four categories (daily, daily_hybrid, hourly, hourly_hybrid)
# at first just within the categories compared to the best model from the modelling notebook
# So for daily that would be linear_order_2, but also the non linear base model and the naive base for comparison

In [ ]:
# Set up parameters for test_hyperparams
daily_lags = daily_lags
used_hourly_lags = used_hourly_lags
ts_daily_train = ts_daily_train
ts_daily_test = ts_daily_test
ts_hourly_train = ts_hourly_train
ts_hourly_test = ts_hourly_test
daily_steps = config["hyperparameter_tuning"]["daily_steps"]
hourly_steps = config["hyperparameter_tuning"]["hourly_steps"]

In [ ]:
# Run test_hyperparams (if you are using the GPU make sure to restart the kernel first and skip the hyperparam tuning cell, otherwise XGBoost may run out of memory depending on your GPU)
test_hyperparams(dict_full, daily_lags, used_hourly_lags, ts_daily_train, ts_daily_test, ts_hourly_train, ts_hourly_test, daily_steps, hourly_steps)